This notebook trains a simple linear Ridge regression model on the running dataset stored in the SQLite database.

Features used: distance (meters), age, year, sex (0/1). Target: time in seconds.

In [40]:
import sqlite3
import pandas as pd
import numpy as np

DB_PATH = "/Users/darraghdonnelly/dev/Database/recovered.db"
TEST_SIZE = 0.20

# connect to db
with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql_query(
        f"SELECT src, sex, age, distance_m, time_s FROM concat_results",
        conn,
    )

print(f"rows: {len(df)}")
df.head()

rows: 143442


,src,sex,age,distance_m,time_s
0,CherryBlossom2017,1,21,16093,3217.0
1,CherryBlossom2017,1,22,16093,3232.0
2,CherryBlossom2017,1,31,16093,3276.0
3,CherryBlossom2017,1,33,16093,3285.0
4,CherryBlossom2017,1,35,16093,3288.0


In [41]:
# split dataset into three buckets by distances
df["distance_bucket"] = (
    df["distance_m"].round().astype(int)
    .map({42195: "42k", 16093: "16k", 5000: "5k"})
)

df["distance_bucket"].head

<bound method NDFrame.head of 0         16k
1         16k
2         16k
3         16k
4         16k
         ... 
143437    16k
143438    16k
143439    16k
143440    16k
143441    16k
Name: distance_bucket, Length: 143442, dtype: str>

In [42]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

# split dataset into train and test - stratified by distance
train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=42, stratify=df["distance_bucket"])

# store features in list
features = ["distance_m", "age", "sex"]

X_train = train_df[features]

# set target to time
y_train = train_df["time_s"]

# convert distances to labeled buckets (5k, 16k, 42k) so stratified k folding can be done
train_buckets = train_df["distance_bucket"]

X_test = test_df[features]
y_test = test_df["time_s"]

model = Ridge(alpha=1.0)

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# get mean abs error scores
cv_mae = -cross_val_score(
    model,
    X_train,
    y_train,
    cv = skf.split(X_train, train_buckets),     # carry out strat kfold on training data for the 3 distances
    scoring="neg_mean_absolute_error"
)

# get r sqrd scores
cv_r2 = cross_val_score(
    model,
    X_train,
    y_train,
    cv=skf.split(X_train, train_buckets),
    scoring="r2"
)
# print results
print("CV MAE:", cv_mae.mean(), "with standard deviation of ±", cv_mae.std())
print("CV R2:", cv_r2.mean(), "with standard deviation of ±", cv_r2.std())

# fit model and run on test once
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Test R2:", r2_score(y_test, y_pred))

CV MAE: 2260.9462230080153 with standard deviation of ± 8.217446382702345
CV R2: 0.7207525804573517 with standard deviation of ± 0.0010935799232115735
Test R2: 0.7191349100645855


In [43]:
from sklearn.metrics import mean_absolute_error

test_eval = test_df.copy()
test_eval["pred_time_s"] = y_pred
test_eval["actual_time_s"] = y_test.values

for bucket in ["5k", "16k", "42k"]:
    bucket_rows = test_eval[test_eval["distance_bucket"] == bucket]
    mae_seconds = mean_absolute_error(bucket_rows["actual_time_s"], bucket_rows["pred_time_s"])
    mae_minutes = mae_seconds / 60

    print(bucket)
    print("MAE seconds:", mae_seconds)
    print("MAE minutes:", mae_minutes)
    print()



5k
MAE seconds: 972.2500545002731
MAE minutes: 16.20416757500455

16k
MAE seconds: 913.7340693181297
MAE minutes: 15.228901155302163

42k
MAE seconds: 2736.4513909659295
MAE minutes: 45.60752318276549



In [44]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

# set features and targets
features = ["distance_m", "age", "sex"]
X = df[features]
y = df["time_s"].astype(float)

# split data into train test using sklearn
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=42
)


# train a simple linear ridge regression model
model = Ridge(alpha=1.0, random_state=42)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# model metrics
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"samples: {len(df)}")
print(f"MAE:  {mae:.2f} s")
print(f"R2:   {r2:.4f}")

# time conversions
distance_to_time = model.coef_[0] * 1000 / 60
age_to_time = model.coef_[1] / 60
sex_to_time = model.coef_[2] / 60

# basic interpretations
print("\nCoefficients (added time per unit of feature):")

print(f"\nPrediction time per added meter: +{distance_to_time:.6f} min/km")
print(f"Per year added to Age: +{age_to_time:.6f} min/year")
print(f"Sex: +{sex_to_time:.6f} min difference (male vs female)")
print(f"intercept: {model.intercept_:.6f}")

samples: 143442
MAE:  2274.79 s
R2:   0.7167

Coefficients (added time per unit of feature):

Prediction time per added meter: +6.820760 min/km
Per year added to Age: +1.080355 min/year
Sex: +28.443940 min difference (male vs female)
intercept: -4094.416616


In [45]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

# convert distance and time (target) to log scale for Reigel model, store in dataframes
df["log_distance"] = np.log(df["distance_m"].astype(float))
y_log = np.log(df["time_s"].astype(float))

# get aged squared so nonliner relationship can be represented
df["ageSqrd"] = df["age"] ** 2

# set features and targets for log model
X = df[["log_distance", "age", "ageSqrd", "sex"]]

# split data into train test
X_train, X_test, y_train, y_test = train_test_split(
    X, y_log, test_size=TEST_SIZE, random_state=42
)

model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

log_pred = model.predict(X_test)

# predict time in secs 
pred_log = model.predict(X_test)

# convert log predictions to seconds by getting the exponential
mae_s = round(mean_absolute_error(np.exp(y_test), np.exp(pred_log)), 2)

print("MAE seconds:", mae_s, "->", round(mae_s/60, 2), "min")
print("coefficients:", dict(zip(X.columns, model.coef_)))
print("intercept:", model.intercept_)

MAE seconds: 2189.45 -> 36.49 min
coefficients: {'log_distance': np.float64(1.0377171327549934), 'age': np.float64(-0.007087284596855199), 'ageSqrd': np.float64(0.00013585988128208014), 'sex': np.float64(0.1255707821557215)}
intercept: -1.377470668200548


In [46]:

# prediction function that reads distance, age and sex - returns time in secs
def predict_time_seconds(distance_m: float, age: float, sex: int) -> float:
    x = pd.DataFrame([{
        "log_distance": np.log(float(distance_m)),
        "age": float(age),
        "ageSqrd": float(age ** 2),
        "sex": float(sex)
    }])
    return float(np.exp(model.predict(x)[0]))

# func to format time into hh:mm:ss
def format_time(seconds: float) -> str:
    s = int(round(seconds))
    h = s // 3600
    m = (s % 3600) // 60
    sec = s % 60
    return f"{h}:{m:02d}:{sec:02d}"

# different prediction examples
Eg1 = format_time(predict_time_seconds(5000, 20, 0))
Eg2 = format_time(predict_time_seconds(16093, 30, 1))
Eg3 = format_time(predict_time_seconds(42195, 30, 0))

young = format_time(predict_time_seconds(5000, 15, 0))
old = format_time(predict_time_seconds(5000, 60, 0))

print("5k male 20:", Eg1)
print("10 mile female 30:", Eg2)
print("Marathon male 30:", Eg3)

5k male 20: 0:26:33
10 mile female 30: 1:40:59
Marathon male 30: 4:02:09


In [47]:
import joblib
from pathlib import Path

# save model as an artifact for use on cold starts
artifact = {
    "kind": "log_ridge",
    "feature_order": ["log_distance", "age", "ageSqrd", "sex"],
    "model": model,
}

out_path = Path("/Users/darraghdonnelly/dev/fyp-running-app/backend/ml_models/base_ridge.joblib")
out_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(artifact, out_path)
print("Saved:", out_path)


Saved: /Users/darraghdonnelly/dev/fyp-running-app/backend/ml_models/base_ridge.joblib


In [48]:
# 5K predictions
fiveK20M = format_time(predict_time_seconds(5000, 20, 0))
fiveK30M = format_time(predict_time_seconds(5000, 30, 0))
fiveK40M = format_time(predict_time_seconds(5000, 40, 0))
fiveK50M = format_time(predict_time_seconds(5000, 50, 0))

fiveK20F = format_time(predict_time_seconds(5000, 20, 1))
fiveK30F = format_time(predict_time_seconds(5000, 30, 1))
fiveK40F = format_time(predict_time_seconds(5000, 40, 1))
fiveK50F = format_time(predict_time_seconds(5000, 50, 1))

# 10 mile predictions
tenMile20M = format_time(predict_time_seconds(16093, 20, 0))
tenMile30M = format_time(predict_time_seconds(16093, 30, 0))
tenMile40M = format_time(predict_time_seconds(16093, 40, 0))
tenMile50M = format_time(predict_time_seconds(16093, 50, 0))

tenMile20F = format_time(predict_time_seconds(16093, 20, 1))
tenMile30F = format_time(predict_time_seconds(16093, 30, 1))
tenMile40F = format_time(predict_time_seconds(16093, 40, 1))
tenMile50F = format_time(predict_time_seconds(16093, 50, 1))

# Marathon predictions
marathon20M = format_time(predict_time_seconds(42195, 20, 0))
marathon30M = format_time(predict_time_seconds(42195, 30, 0))
marathon40M = format_time(predict_time_seconds(42195, 40, 0))
marathon50M = format_time(predict_time_seconds(42195, 50, 0))

marathon20F = format_time(predict_time_seconds(42195, 20, 1))
marathon30F = format_time(predict_time_seconds(42195, 30, 1))
marathon40F = format_time(predict_time_seconds(42195, 40, 1))
marathon50F = format_time(predict_time_seconds(42195, 50, 1))

print("\n5K:")
print("5k male 20:", fiveK20M, "vs female 20:", fiveK20F)
print("5k male 30:", fiveK30M, "vs female 30:", fiveK30F)
print("5K male 40:", fiveK40M, "vs female 40:", fiveK40F)
print("5K male 50:", fiveK50M, "vs female 50:", fiveK50F)

print("\n10 Mile")
print("Male 20:", tenMile20M, "vs female 20:", tenMile20F)
print("Male 30:", tenMile30M, "vs female 30:", tenMile30F)
print("Male 40:", tenMile40M, "vs female 40:", tenMile40F)
print("Male 50:", tenMile50M, "vs female 50:", tenMile50F)

print("\nMarathon")
print("Male 20:", marathon20M, "vs female 20:", marathon20F)
print("Male 30:", marathon30M, "vs female 30:", marathon30F)
print("Male 40:", marathon40M, "vs female 40:", marathon40F)
print("Male 50:", marathon50M, "vs female 50:", marathon50F)


5K:
5k male 20: 0:26:33 vs female 20: 0:30:06
5k male 30: 0:26:29 vs female 30: 0:30:01
5K male 40: 0:27:08 vs female 40: 0:30:45
5K male 50: 0:28:33 vs female 50: 0:32:23

10 Mile
Male 20: 1:29:19 vs female 20: 1:41:16
Male 30: 1:29:04 vs female 30: 1:40:59
Male 40: 1:31:15 vs female 40: 1:43:27
Male 50: 1:36:03 vs female 50: 1:48:55

Marathon
Male 20: 4:02:52 vs female 20: 4:35:22
Male 30: 4:02:09 vs female 30: 4:34:33
Male 40: 4:08:06 vs female 40: 4:41:17
Male 50: 4:21:11 vs female 50: 4:56:08
